# Corpus Summary Report Generator
Generates a `.docx` pre-processing summary report from `corpus.csv`.

In [3]:
# pip install python-docx

In [4]:
# ── Imports ────────────────────────────────────────────────────────────────
# pandas for loading and grouping the CSV.
# re for simple regex-based tokenization.
# python-docx for building the Word document and styling tables.
import pandas as pd
import re
from docx import Document
from docx.shared import Pt, RGBColor
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

In [5]:
# ── Stopwords list ─────────────────────────────────────────────────────────
# A basic set of common English function words used to compute
# the stopword percentage statistic in Section 4.
# No external library needed — just a plain Python set.
STOPWORDS = set(
    "a about above after again all also am an and any are as at be because "
    "been before being below between both but by can did do does doing down "
    "during each few for from further get had has have having he her here him "
    "his how i if in into is it its itself just me more most my myself no nor "
    "not now of off on once only or other our out over own same she should so "
    "some such than that the their them then there these they this those through "
    "to too under until up very was we were what when where which while who whom "
    "why will with would you your".split()
)

In [6]:
# ── Tokenizer ──────────────────────────────────────────────────────────────
# Lowercases the text and extracts only alphabetic words.
# Strips punctuation, numbers, and special characters.
def tokenize(text):
    return re.findall(r'\b[a-z]+\b', (text or '').lower())

In [8]:
# ── Load dataset ───────────────────────────────────────────────────────────
# Reads the CSV into a DataFrame. Expected columns:
# title, text, url, content_type, topic, word_count
df = pd.read_csv('corpus.csv')

In [9]:
# ── Compute per-topic vocabulary statistics ────────────────────────────────
# For each topic, concatenate all document tokens and compute:
#   count     – number of documents
#   avg_words – mean word count (from the word_count column)
#   vocab     – number of unique token types
#   ttr       – Type-Token Ratio: unique types / total tokens
#   pct_stop  – percentage of tokens that are stopwords
topic_stats = {}
for topic, grp in df.groupby('topic'):
    tokens = [t for text in grp['text'] for t in tokenize(text)]
    vocab = set(tokens)
    stops = sum(1 for t in tokens if t in STOPWORDS)
    topic_stats[topic] = {
        'count':     len(grp),
        'avg_words': round(grp['word_count'].mean()),
        'vocab':     len(vocab),
        'ttr':       round(len(vocab) / len(tokens), 4) if tokens else 0,
        'pct_stop':  round(stops / len(tokens) * 100, 1) if tokens else 0,
        '_vocab_set': vocab,
    }

topics = sorted(topic_stats)

In [11]:
# ── Compute pairwise lexical overlap (Jaccard similarity) ──────────────────
# For every pair of topics, compute Jaccard = |A ∩ B| / |A ∪ B|
# where A and B are each topic's raw vocabulary sets.
# Results are sorted descending so the most similar pairs appear first.
overlap_rows = []
for i, t1 in enumerate(topics):
    for t2 in topics[i+1:]:
        a, b = topic_stats[t1]['_vocab_set'], topic_stats[t2]['_vocab_set']
        j = round(len(a & b) / len(a | b), 3)
        overlap_rows.append((f'{t1} \u2194 {t2}', j))
overlap_rows.sort(key=lambda x: -x[1])

In [12]:
# ── Table builder helper ───────────────────────────────────────────────────
# Adds a styled table to the document with:
#   - Blue header row with white bold text
#   - Alternating row shading (light blue / white)
#   - Column widths specified in centimetres
def add_table(doc, headers, rows, col_widths_cm):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Table Grid'

    # Header row
    hdr = table.rows[0]
    for i, h in enumerate(headers):
        cell = hdr.cells[i]
        cell.width = int(col_widths_cm[i] * 360000)
        cell.paragraphs[0].clear()
        run = cell.paragraphs[0].add_run(h)
        run.bold = True
        run.font.size = Pt(9)
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        tcPr = cell._tc.get_or_add_tcPr()
        shd = OxmlElement('w:shd')
        shd.set(qn('w:fill'), '2E74B5')
        shd.set(qn('w:val'), 'clear')
        tcPr.append(shd)

    # Data rows
    for ri, row_data in enumerate(rows):
        row = table.add_row()
        fill = 'EBF3FB' if ri % 2 == 0 else 'FFFFFF'
        for i, val in enumerate(row_data):
            cell = row.cells[i]
            cell.width = int(col_widths_cm[i] * 360000)
            cell.paragraphs[0].clear()
            cell.paragraphs[0].add_run(str(val)).font.size = Pt(9)
            tcPr = cell._tc.get_or_add_tcPr()
            shd = OxmlElement('w:shd')
            shd.set(qn('w:fill'), fill)
            shd.set(qn('w:val'), 'clear')
            tcPr.append(shd)

In [13]:
# ── Initialise document and set default font ───────────────────────────────
# Creates a blank Word document and sets Arial 10pt as the base style
# so all text inherits it unless explicitly overridden.
doc = Document()
for s in doc.styles:
    if s.name == 'Normal':
        s.font.name = 'Arial'
        s.font.size = Pt(10)

total = len(df)
avg   = round(df['word_count'].mean())

doc.add_heading('Corpus Summary Report', 0)
doc.add_paragraph(
    f'Total: {total} docs  |  Topics: {len(topics)}  |  '
    f'Types: {df["content_type"].nunique()}  |  Overall avg length: {avg} words'
)

In [14]:
# ── Section 1: Document type distribution ─────────────────────────────────
# Counts documents per content_type and expresses each as a
# percentage of the total corpus size.
doc.add_heading('1. Document Type Distribution', 2)
ct = df['content_type'].value_counts()
add_table(
    doc,
    ['Content Type', 'Count', '% of Corpus'],
    [(k, v, f'{v/total*100:.1f}%') for k, v in ct.items()],
    [9, 3, 4]
)

In [15]:
# ── Section 2: Topic distribution ─────────────────────────────────────────
# Counts documents per topic and expresses each as a
# percentage of the total corpus size.
doc.add_heading('2. Topic Distribution', 2)
add_table(
    doc,
    ['Topic', 'Count', '% of Corpus'],
    [(t, topic_stats[t]['count'], f'{topic_stats[t]["count"]/total*100:.1f}%') for t in topics],
    [9, 3, 4]
)

In [16]:
# ── Section 3: Average document length per topic ───────────────────────────
# Uses the pre-computed avg_words value (mean of word_count column)
# for each topic group.
doc.add_heading('3. Avg Document Length per Topic', 2)
add_table(
    doc,
    ['Topic', 'Docs', 'Avg Words'],
    [(t, topic_stats[t]['count'], topic_stats[t]['avg_words']) for t in topics],
    [9, 3, 4]
)

In [17]:
# ── Section 4: Vocabulary statistics per topic ────────────────────────────
# Reports three vocabulary metrics per topic:
#   Vocab Size – total unique word types in the topic
#   TTR        – Type-Token Ratio (lexical diversity)
#   % Stopwords – noise load before any preprocessing
doc.add_heading('4. Vocabulary Statistics per Topic', 2)
add_table(
    doc,
    ['Topic', 'Vocab Size', 'TTR', '% Stopwords'],
    [(t, topic_stats[t]['vocab'], topic_stats[t]['ttr'], f'{topic_stats[t]["pct_stop"]}%') for t in topics],
    [7, 3.5, 3, 2.5]
)

In [18]:
# ── Section 5: Lexical overlap between topics ─────────────────────────────
# Lists all 28 topic pairs ranked by Jaccard similarity.
# Higher scores mean more shared vocabulary between the two topics.
doc.add_heading('5. Lexical Overlap Between Topics (Jaccard)', 2)
add_table(
    doc,
    ['Topic Pair', 'Jaccard'],
    overlap_rows,
    [13, 3]
)

In [19]:
# ── Save document ──────────────────────────────────────────────────────────
# Writes the completed document to disk.
doc.save('corpus_summary_report.docx')
print('Done — corpus_summary_report.docx saved.')

Done — corpus_summary_report.docx saved.
